In [ ]:
import numpy as np
import pandas as pd
import warnings

# 警告を非表示
warnings.filterwarnings('ignore')

# ==================================
# 1. パラメータ設定
# ==================================
# 動作確認用に規模を縮小
N_COHORT = 10000  # 100,000 -> 10,000
START_YEAR = 2026
SIM_YEARS = 65
N_ITERATIONS = 2  # 10 -> 2

# 表2.1: 年齢階級別人口構成比 (20-84歳)
AGE_GROUPS = np.arange(20, 85, 5)
AGE_DIST = np.array([7.9, 8.3, 8.3, 8.1, 8.9, 10.1, 12.0, 8.6, 7.3, 7.6, 8.8, 9.7, 7.1])
AGE_DIST = AGE_DIST / np.sum(AGE_DIST)

# 表2.2: 保険制度別コホート割合 (30-64歳)
INSURANCE_DIST = {
    30: np.array([25.3, 3.9, 70.8]) / 100,
    35: np.array([26.9, 6.2, 66.9]) / 100,
    40: np.array([24.4, 6.5, 69.1]) / 100,
    45: np.array([24.7, 6.9, 68.4]) / 100,
    50: np.array([22.8, 6.8, 70.4]) / 100,
    55: np.array([22.9, 7.1, 70.0]) / 100,
    60: np.array([16.7, 5.5, 77.8]) / 100
}

# 表3.1-3.6 疫学関数群
def get_h_pylori_prob(birth_year):
    years = [1950, 1960, 1970, 1980, 1990, 2000, 2010]
    probs = [59.1, 49.1, 34.9, 24.6, 15.6, 6.6, 3.0]
    return np.interp(birth_year, years, probs) / 100.0

def get_severe_atrophy_prob(age):
    ages = [40, 50, 60, 70]
    probs = [0.0, 11.1, 41.5, 57.9]
    return np.interp(age, ages, probs) / 100.0

STAGE_DIST_SCREEN = np.array([76.5, 2.9, 4.4, 13.2, 2.9]) / 100.0
STAGE_DIST_SYMPTOM = np.array([29.9, 10.2, 18.9, 40.9, 0.8]) / 100.0
STAGE_DIST_SCREEN = STAGE_DIST_SCREEN / np.sum(STAGE_DIST_SCREEN)
STAGE_DIST_SYMPTOM = STAGE_DIST_SYMPTOM / np.sum(STAGE_DIST_SYMPTOM)
STAGES = ['I', 'II', 'III', 'IV', 'Unknown']

SURVIVAL_5YR = {'I': 0.928, 'II': 0.666, 'III': 0.414, 'IV': 0.067, 'Unknown': 0.500}
ANNUAL_CANCER_DEATH_PROB = {k: 1.0 - (v ** 0.2) for k, v in SURVIVAL_5YR.items()}

ANNUAL_INCIDENCE = {
    'Male':   {'Infected': 0.170 / 65, 'Eradicated': 0.057 / 65, 'Uninfected': 0.010 / 65},
    'Female': {'Infected': 0.077 / 65, 'Eradicated': 0.0257 / 65, 'Uninfected': 0.0045 / 65}
}

def get_general_death_prob(age, sex):
    age_idx = min(max((age - 20) // 10, 0), 8)
    rates_m = [0.049, 0.078, 0.150, 0.318, 0.693, 1.423, 3.819, 12.932, 36.830]
    rates_f = [0.024, 0.038, 0.071, 0.146, 0.300, 0.688, 2.251, 9.172, 31.294]
    rates = rates_m if sex == 'Male' else rates_f
    return rates[age_idx] / 100.0

# 経済評価用ベースパラメータ (PSAで変動させるベース値)
COST_PYLORI_TEST = 3000
COST_ERADICATION = 10000
BASE_COST_SCREENING = 15000
BASE_COST_CANCER_TRT = { 'I': 500000, 'II': 1000000, 'III': 2000000, 'IV': 3000000, 'Unknown': 1000000 }
BASE_QALY_WEIGHTS = { 'Healthy': 1.0, 'Cancer_I': 0.8, 'Cancer_II': 0.7, 'Cancer_III': 0.6, 'Cancer_IV': 0.4, 'Cancer_Unknown': 0.6 }

# ==========================================
# ★PSA追加: パラメータの確率分布サンプリング関数
# ==========================================
def sample_psa_parameters():
    """毎回のイテレーションで確率分布からパラメータを取得する"""
    psa_params = {}

    # 1. 行動変容の不確実性 (Beta分布: 0〜1の割合)
    # 戦略Sの受診率 (中心50%, a=50, b=50 -> 概ね40〜60%の間に分布)
    psa_params['rate_screen_S'] = np.random.beta(50, 50)
    # 職域検診での除菌実施率 (中心55%, a=55, b=45 -> 概ね45〜65%の間に分布)
    psa_params['rate_erad_implement'] = np.random.beta(55, 45)

    # 2. コストの不確実性 (Gamma分布: 正の値で右裾が長い分布)
    # 内視鏡検査コスト (平均15000, SD=3000)
    shape_scr = (15000 / 3000) ** 2
    scale_scr = 15000 / shape_scr
    psa_params['cost_screening'] = np.random.gamma(shape_scr, scale_scr)

    # がん治療コスト(I期) (平均500,000, SD=100,000)
    shape_trt1 = (500000 / 100000) ** 2
    scale_trt1 = 500000 / shape_trt1
    psa_params['cost_cancer_I'] = np.random.gamma(shape_trt1, scale_trt1)

    # 3. QALY効用値の不確実性 (Beta分布: 0〜1のスコア)
    # がんI期のQALYウェイト (平均0.8, SD=0.05 -> a≒51.2, b≒12.8)
    psa_params['qaly_cancer_I'] = np.random.beta(51.2, 12.8)

    return psa_params

# ==========================================
# 2. コホート初期化関数
# ==========================================
def initialize_cohort(sex, strategy_p=False):
    df = pd.DataFrame({
        'id': np.arange(N_COHORT),
        'age': np.random.choice(AGE_GROUPS, size=N_COHORT, p=AGE_DIST) + np.random.randint(0, 5, size=N_COHORT),
        'sex': np.full(N_COHORT, sex, dtype='object'),
        'insurance': np.full(N_COHORT, 'Unknown', dtype='object'),
        'nhi_old_rule': np.random.rand(N_COHORT) < 0.30,
        'pylori_status': np.full(N_COHORT, 'Uninfected', dtype='object'),
        'atrophy': np.full(N_COHORT, 'None', dtype='object'),
        'cancer_status': np.full(N_COHORT, 'None', dtype='object'),
        'cancer_stage': np.full(N_COHORT, 'None', dtype='object'),
        'is_dead': np.zeros(N_COHORT, dtype=bool),
        'death_cause': np.full(N_COHORT, 'None', dtype='object'),
        'years_lived': np.zeros(N_COHORT, dtype=float),
        'total_cost': np.zeros(N_COHORT, dtype=float),
        'total_qaly': np.zeros(N_COHORT, dtype=float),
        'screen_count': np.zeros(N_COHORT, dtype=int),
        'true_positive': np.zeros(N_COHORT, dtype=int),
        'false_positive': np.zeros(N_COHORT, dtype=int),
        'last_screen_age': np.full(N_COHORT, -1, dtype=int)
    })

    for base_age, probs in INSURANCE_DIST.items():
        mask = (df['age'] >= base_age) & (df['age'] < base_age + 5)
        n_mask = mask.sum()
        if n_mask > 0:
            p_norm = probs / np.sum(probs)
            df.loc[mask, 'insurance'] = np.random.choice(['Employee', 'Dependent', 'NHI'], size=n_mask, p=p_norm)

    birth_years = START_YEAR - df['age']
    inf_probs = np.array([get_h_pylori_prob(y) for y in birth_years])
    df.loc[np.random.rand(N_COHORT) < inf_probs, 'pylori_status'] = 'Infected'

    inf_mask = df['pylori_status'] == 'Infected'
    sev_probs = np.array([get_severe_atrophy_prob(a) for a in df['age']])
    rand_vals = np.random.rand(N_COHORT)

    severe_mask = inf_mask & (rand_vals < sev_probs)
    mild_mask = inf_mask & (rand_vals >= sev_probs)

    df.loc[severe_mask, 'atrophy'] = 'Severe'
    df.loc[mild_mask, 'atrophy'] = 'Mild'

    if strategy_p:
        eradicate_mask = inf_mask & (np.random.rand(N_COHORT) < 0.90)
        df.loc[eradicate_mask, 'pylori_status'] = 'Eradicated'
        df.loc[eradicate_mask, 'total_cost'] += COST_PYLORI_TEST + COST_ERADICATION
        df.loc[(df['age'] >= 20) & (~eradicate_mask), 'total_cost'] += COST_PYLORI_TEST * 0.8

    return df

# ==========================================
# 3. シミュレーション実行関数 (PSA対応)
# ==========================================
def run_simulation(cohort, strategy_name, psa_params): # ★引数にpsa_paramsを追加
    df = cohort.copy()

    # ★PSAパラメータの展開
    rate_screen_S = psa_params['rate_screen_S']
    rate_erad_impl = psa_params['rate_erad_implement']
    cost_screening = psa_params['cost_screening']
    qaly_w_cancer_I = psa_params['qaly_cancer_I']

    cost_cancer_trt = BASE_COST_CANCER_TRT.copy()
    cost_cancer_trt['I'] = psa_params['cost_cancer_I']

    for year in range(SIM_YEARS):
        current_age = df['age']
        alive_mask = ~df['is_dead']

        if not alive_mask.any():
            break

        gen_death_probs = np.array([get_general_death_prob(a, df['sex'].iloc[0]) for a in current_age])
        gen_death_mask = alive_mask & (np.random.rand(N_COHORT) < gen_death_probs)
        df.loc[gen_death_mask, 'is_dead'] = True
        df.loc[gen_death_mask, 'death_cause'] = 'General'
        alive_mask = ~df['is_dead']

        diag_mask = alive_mask & (df['cancer_status'] == 'Diagnosed')
        for stage in STAGES:
            stage_mask = diag_mask & (df['cancer_stage'] == stage)
            death_mask = stage_mask & (np.random.rand(N_COHORT) < ANNUAL_CANCER_DEATH_PROB[stage])
            df.loc[death_mask, 'is_dead'] = True
            df.loc[death_mask, 'death_cause'] = 'GastricCancer'
        alive_mask = ~df['is_dead']

        risk_none = alive_mask & (df['cancer_status'] == 'None')
        for status in ['Infected', 'Eradicated', 'Uninfected']:
            status_mask = risk_none & (df['pylori_status'] == status)
            inc_prob = ANNUAL_INCIDENCE[df['sex'].iloc[0]][status]
            inc_mask = status_mask & (np.random.rand(N_COHORT) < inc_prob)
            df.loc[inc_mask, 'cancer_status'] = 'Undiagnosed'

        screen_probs = np.zeros(N_COHORT)
        if strategy_name == 'H':
            emp_mask = alive_mask & (df['insurance'] == 'Employee') & (current_age >= 40) & (current_age <= 64)
            dep_mask = alive_mask & (df['insurance'] == 'Dependent') & (current_age >= 50) & (current_age <= 64) & ((current_age - df['last_screen_age']) >= 2)
            nhi_new_mask = alive_mask & (df['insurance'] == 'NHI') & (~df['nhi_old_rule']) & (current_age >= 50) & (current_age <= 64) & ((current_age - df['last_screen_age']) >= 2)
            nhi_old_mask = alive_mask & (df['insurance'] == 'NHI') & (df['nhi_old_rule']) & (current_age >= 40) & (current_age <= 64)
            over65_mask = alive_mask & (current_age >= 65) & ((current_age - df['last_screen_age']) >= 2)

            screen_probs[emp_mask] = 0.75
            screen_probs[dep_mask] = 0.45
            screen_probs[nhi_new_mask] = 0.35
            screen_probs[nhi_old_mask] = 0.35
            screen_probs[over65_mask] = 0.35
        elif strategy_name == 'S':
            target_mask = alive_mask & (current_age >= 50) & ((current_age - df['last_screen_age']) >= 2)
            screen_probs[target_mask] = rate_screen_S # ★PSAパラメータを使用
        elif strategy_name == 'P':
            sev_mask = alive_mask & (df['pylori_status'].isin(['Infected', 'Eradicated'])) & (df['atrophy'] == 'Severe')
            mild_mask = alive_mask & (df['pylori_status'].isin(['Infected', 'Eradicated'])) & (df['atrophy'] == 'Mild') & ((current_age - df['last_screen_age']) >= 2)
            screen_probs[sev_mask] = 0.75
            screen_probs[mild_mask] = 0.75

        do_screen_mask = alive_mask & (np.random.rand(N_COHORT) < screen_probs)
        df.loc[do_screen_mask, 'screen_count'] += 1
        df.loc[do_screen_mask, 'last_screen_age'] = current_age
        df.loc[do_screen_mask, 'total_cost'] += cost_screening # ★PSAパラメータを使用

        if strategy_name == 'H':
            # ★PSAパラメータ(rate_erad_impl)を使用
            h_erad_mask = do_screen_mask & emp_mask & (df['pylori_status'] == 'Infected') & (np.random.rand(N_COHORT) < rate_erad_impl * 0.90)
            df.loc[h_erad_mask, 'pylori_status'] = 'Eradicated'
            df.loc[h_erad_mask, 'total_cost'] += COST_ERADICATION

        undiag_mask = alive_mask & (df['cancer_status'] == 'Undiagnosed')
        screen_found_mask = undiag_mask & do_screen_mask
        if screen_found_mask.any():
            df.loc[screen_found_mask, 'cancer_status'] = 'Diagnosed'
            df.loc[screen_found_mask, 'true_positive'] += 1
            n_found = screen_found_mask.sum()
            df.loc[screen_found_mask, 'cancer_stage'] = np.random.choice(STAGES, size=n_found, p=STAGE_DIST_SCREEN)

        fp_mask = alive_mask & (df['cancer_status'] == 'None') & do_screen_mask & (np.random.rand(N_COHORT) < 0.05)
        df.loc[fp_mask, 'false_positive'] += 1

        symp_found_mask = undiag_mask & (~do_screen_mask) & (np.random.rand(N_COHORT) < 0.20)
        if symp_found_mask.any():
            df.loc[symp_found_mask, 'cancer_status'] = 'Diagnosed'
            n_found = symp_found_mask.sum()
            df.loc[symp_found_mask, 'cancer_stage'] = np.random.choice(STAGES, size=n_found, p=STAGE_DIST_SYMPTOM)

        alive_mask = ~df['is_dead']
        df.loc[alive_mask, 'years_lived'] += 1
        df.loc[alive_mask, 'age'] += 1

        qaly_vals = np.ones(N_COHORT) * BASE_QALY_WEIGHTS['Healthy']
        for stage in STAGES:
            stage_mask = alive_mask & (df['cancer_status'] == 'Diagnosed') & (df['cancer_stage'] == stage)
            # ★PSAパラメータを使用
            q_weight = qaly_w_cancer_I if stage == 'I' else BASE_QALY_WEIGHTS[f'Cancer_{stage}']
            qaly_vals[stage_mask] = q_weight
            df.loc[stage_mask, 'total_cost'] += cost_cancer_trt[stage] / 5.0

        df.loc[alive_mask, 'total_qaly'] += qaly_vals[alive_mask]

    return df

# ==========================================
# 4. 反復シミュレーション用の指標計算と集計
# ==========================================
def calculate_iteration_metrics(results_dict, psa_params):
    metrics = {}
    # ★実行時のPSAパラメータを記録（散布図などの分析用）
    for k, v in psa_params.items():
        metrics[f'Param_{k}'] = v

    for st, df in results_dict.items():
        gc_deaths = (df['death_cause'] == 'GastricCancer').sum()
        person_years = df['years_lived'].sum()
        avg_cost = df['total_cost'].mean()
        avg_qaly = df['total_qaly'].mean()

        metrics[f'{st}_Deaths'] = gc_deaths
        metrics[f'{st}_PersonYears'] = person_years
        metrics[f'{st}_Cost'] = avg_cost
        metrics[f'{st}_QALY'] = avg_qaly
        metrics[f'{st}_Screens'] = df['screen_count'].sum()

    base_deaths = metrics['H_Deaths']
    base_cost = metrics['H_Cost']
    base_qaly = metrics['H_QALY']
    base_py = metrics['H_PersonYears']

    for st in ['S', 'P']:
        st_deaths = metrics[f'{st}_Deaths']
        st_cost = metrics[f'{st}_Cost']
        st_qaly = metrics[f'{st}_QALY']
        st_py = metrics[f'{st}_PersonYears']

        rr = (st_deaths / N_COHORT) / (base_deaths / N_COHORT) if base_deaths > 0 else np.nan
        metrics[f'{st}_vs_H_RR'] = rr

        rate_st = st_deaths / st_py if st_py > 0 else 0
        rate_h = base_deaths / base_py if base_py > 0 else 0
        hr_approx = rate_st / rate_h if rate_h > 0 else np.nan
        metrics[f'{st}_vs_H_HR_approx'] = hr_approx

        dc = st_cost - base_cost
        dq = st_qaly - base_qaly
        icer = dc / dq if dq > 0 else np.nan
        metrics[f'{st}_vs_H_ICER'] = icer
        metrics[f'{st}_vs_H_dCost'] = dc
        metrics[f'{st}_vs_H_dQALY'] = dq

    return metrics

def summarize_iterations(metrics_list, sex):
    df_res = pd.DataFrame(metrics_list)

    print(f"\n{'='*60}")
    print(f" PSA(確率的感度分析) 集計結果: {sex}")
    print(f" (コホート: {N_COHORT:,}人, サンプリング反復回数: {N_ITERATIONS}回)")
    print(f"{'='*60}")

    print("\n--- 胃がん死亡数 (平均 [95%信用区間]) ---")
    for st in ['H', 'S', 'P']:
        mean_val = df_res[f'{st}_Deaths'].mean()
        p025 = df_res[f'{st}_Deaths'].quantile(0.025)
        p975 = df_res[f'{st}_Deaths'].quantile(0.975)
        print(f"戦略{st}: {mean_val:.1f}人 [{p025:.1f} - {p975:.1f}]")

    print("\n--- RR: 相対リスク (平均 [95%信用区間]) ---")
    for st in ['S', 'P']:
        col = f'{st}_vs_H_RR'
        mean_val = df_res[col].mean()
        p025 = df_res[col].quantile(0.025)
        p975 = df_res[col].quantile(0.975)
        print(f"戦略{st} vs H: {mean_val:.3f} [{p025:.3f} - {p975:.3f}]")

    print("\n--- ICER: 増分費用効果比 (中央値 [95%信用区間]) JPY/QALY ---")
    for st in ['S', 'P']:
        icer_col = f'{st}_vs_H_ICER'
        dq_col = f'{st}_vs_H_dQALY'

        valid_icers = df_res[df_res[dq_col] > 0][icer_col].dropna()
        n_dominated = (df_res[dq_col] <= 0).sum()

        if len(valid_icers) > 0:
            median_val = valid_icers.median()
            p025 = valid_icers.quantile(0.025)
            p975 = valid_icers.quantile(0.975)
            print(f"戦略{st} vs H: {median_val:,.0f} [{p025:,.0f} - {p975:,.0f}]")
        else:
            print(f"戦略{st} vs H: 評価不能 (全回で Dominant または Dominated)")

        if n_dominated > 0:
            print(f"  ※ {n_dominated}回/{N_ITERATIONS}回 で戦略{st}はDominant(低コスト・高QALY)またはDominatedでした。")

# ==========================================
# メイン実行ブロック
# ==========================================
if __name__ == "__main__":
    np.random.seed(42) # 全体の再現性を担保

    for sex in ['Male', 'Female']:
        metrics_history = []
        print(f"\n>>> {sex} のPSAシミュレーションを開始します...")

        for i in range(N_ITERATIONS):
            # ★各イテレーションでパラメータを確率分布からサンプリング
            current_psa_params = sample_psa_parameters()

            base_cohort_h_s = initialize_cohort(sex, strategy_p=False)
            base_cohort_p = initialize_cohort(sex, strategy_p=True)

            results = {}
            # サンプリングしたパラメータをシミュレーション関数に渡す
            results['H'] = run_simulation(base_cohort_h_s, 'H', current_psa_params)
            results['S'] = run_simulation(base_cohort_h_s, 'S', current_psa_params)
            results['P'] = run_simulation(base_cohort_p, 'P', current_psa_params)

            # 結果と使用したパラメータを記録
            iteration_metrics = calculate_iteration_metrics(results, current_psa_params)
            iteration_metrics['Iteration'] = i + 1
            metrics_history.append(iteration_metrics)

        summarize_iterations(metrics_history, sex)


>>> Male のPSAシミュレーションを開始します...

 PSA(確率的感度分析) 集計結果: Male
 (コホート: 10,000人, サンプリング反復回数: 2回)

--- 胃がん死亡数 (平均 [95%信用区間]) ---
戦略H: 95.0人 [92.2 - 97.8]
戦略S: 123.0人 [116.3 - 129.7]
戦略P: 65.5人 [58.4 - 72.6]

--- RR: 相対リスク (平均 [95%信用区間]) ---
戦略S vs H: 1.294 [1.263 - 1.325]
戦略P vs H: 0.688 [0.633 - 0.742]

--- ICER: 増分費用効果比 (中央値 [95%信用区間]) JPY/QALY ---
戦略S vs H: 評価不能 (全回で Dominant または Dominated)
  ※ 2回/2回 で戦略SはDominant(低コスト・高QALY)またはDominatedでした。
戦略P vs H: 評価不能 (全回で Dominant または Dominated)
  ※ 2回/2回 で戦略PはDominant(低コスト・高QALY)またはDominatedでした。

>>> Female のPSAシミュレーションを開始します...

 PSA(確率的感度分析) 集計結果: Female
 (コホート: 10,000人, サンプリング反復回数: 2回)

--- 胃がん死亡数 (平均 [95%信用区間]) ---
戦略H: 62.5人 [61.1 - 63.9]
戦略S: 62.5人 [62.0 - 63.0]
戦略P: 31.5人 [25.3 - 37.7]

--- RR: 相対リスク (平均 [95%信用区間]) ---
戦略S vs H: 1.000 [0.985 - 1.016]
戦略P vs H: 0.507 [0.396 - 0.617]

--- ICER: 増分費用効果比 (中央値 [95%信用区間]) JPY/QALY ---
戦略S vs H: 726,006 [354,405 - 1,097,607]
戦略P vs H: -437,890 [-473,249 - -402,531]


In [ ]:
!pip install numpy pandas matplotlib seaborn

In [ ]:
import numpy as np
import pandas as pd
import warnings

# 警告を非表示
warnings.filterwarnings('ignore')

# ==================================
# 1. パラメータ設定
# ==================================
# 動作確認用に規模を縮小
N_COHORT = 10000  # 100,000 -> 10,000
START_YEAR = 2026
SIM_YEARS = 65
N_ITERATIONS = 2  # 10 -> 2

# 表2.1: 年齢階級別人口構成比 (20-84歳)
AGE_GROUPS = np.arange(20, 85, 5)
AGE_DIST = np.array([7.9, 8.3, 8.3, 8.1, 8.9, 10.1, 12.0, 8.6, 7.3, 7.6, 8.8, 9.7, 7.1])
AGE_DIST = AGE_DIST / np.sum(AGE_DIST)

# 表2.2: 保険制度別コホート割合 (30-64歳)
INSURANCE_DIST = {
    30: np.array([25.3, 3.9, 70.8]) / 100,
    35: np.array([26.9, 6.2, 66.9]) / 100,
    40: np.array([24.4, 6.5, 69.1]) / 100,
    45: np.array([24.7, 6.9, 68.4]) / 100,
    50: np.array([22.8, 6.8, 70.4]) / 100,
    55: np.array([22.9, 7.1, 70.0]) / 100,
    60: np.array([16.7, 5.5, 77.8]) / 100
}

# 表3.1-3.6 疫学関数群
def get_h_pylori_prob(birth_year):
    years = [1950, 1960, 1970, 1980, 1990, 2000, 2010]
    probs = [59.1, 49.1, 34.9, 24.6, 15.6, 6.6, 3.0]
    return np.interp(birth_year, years, probs) / 100.0

def get_severe_atrophy_prob(age):
    ages = [40, 50, 60, 70]
    probs = [0.0, 11.1, 41.5, 57.9]
    return np.interp(age, ages, probs) / 100.0

STAGE_DIST_SCREEN = np.array([76.5, 2.9, 4.4, 13.2, 2.9]) / 100.0
STAGE_DIST_SYMPTOM = np.array([29.9, 10.2, 18.9, 40.9, 0.8]) / 100.0
STAGE_DIST_SCREEN = STAGE_DIST_SCREEN / np.sum(STAGE_DIST_SCREEN)
STAGE_DIST_SYMPTOM = STAGE_DIST_SYMPTOM / np.sum(STAGE_DIST_SYMPTOM)
STAGES = ['I', 'II', 'III', 'IV', 'Unknown']

SURVIVAL_5YR = {'I': 0.928, 'II': 0.666, 'III': 0.414, 'IV': 0.067, 'Unknown': 0.500}
ANNUAL_CANCER_DEATH_PROB = {k: 1.0 - (v ** 0.2) for k, v in SURVIVAL_5YR.items()}

ANNUAL_INCIDENCE = {
    'Male':   {'Infected': 0.170 / 65, 'Eradicated': 0.057 / 65, 'Uninfected': 0.010 / 65},
    'Female': {'Infected': 0.077 / 65, 'Eradicated': 0.0257 / 65, 'Uninfected': 0.0045 / 65}
}

def get_general_death_prob(age, sex):
    age_idx = min(max((age - 20) // 10, 0), 8)
    rates_m = [0.049, 0.078, 0.150, 0.318, 0.693, 1.423, 3.819, 12.932, 36.830]
    rates_f = [0.024, 0.038, 0.071, 0.146, 0.300, 0.688, 2.251, 9.172, 31.294]
    rates = rates_m if sex == 'Male' else rates_f
    return rates[age_idx] / 100.0

# 経済評価用ベースパラメータ (PSAで変動させるベース値)
COST_PYLORI_TEST = 3000
COST_ERADICATION = 10000
BASE_COST_SCREENING = 15000
BASE_COST_CANCER_TRT = { 'I': 500000, 'II': 1000000, 'III': 2000000, 'IV': 3000000, 'Unknown': 1000000 }
BASE_QALY_WEIGHTS = { 'Healthy': 1.0, 'Cancer_I': 0.8, 'Cancer_II': 0.7, 'Cancer_III': 0.6, 'Cancer_IV': 0.4, 'Cancer_Unknown': 0.6 }

# ==========================================
# ★PSA追加: パラメータの確率分布サンプリング関数
# ==========================================
def sample_psa_parameters():
    """毎回のイテレーションで確率分布からパラメータを取得する"""
    psa_params = {}

    # 1. 行動変容の不確実性 (Beta分布: 0〜1の割合)
    # 戦略Sの受診率 (中心50%, a=50, b=50 -> 概ね40〜60%の間に分布)
    psa_params['rate_screen_S'] = np.random.beta(50, 50)
    # 職域検診での除菌実施率 (中心55%, a=55, b=45 -> 概ね45〜65%の間に分布)
    psa_params['rate_erad_implement'] = np.random.beta(55, 45)

    # 2. コストの不確実性 (Gamma分布: 正の値で右裾が長い分布)
    # 内視鏡検査コスト (平均15000, SD=3000)
    shape_scr = (15000 / 3000) ** 2
    scale_scr = 15000 / shape_scr
    psa_params['cost_screening'] = np.random.gamma(shape_scr, scale_scr)

    # がん治療コスト(I期) (平均500,000, SD=100,000)
    shape_trt1 = (500000 / 100000) ** 2
    scale_trt1 = 500000 / shape_trt1
    psa_params['cost_cancer_I'] = np.random.gamma(shape_trt1, scale_trt1)

    # 3. QALY効用値の不確実性 (Beta分布: 0〜1のスコア)
    # がんI期のQALYウェイト (平均0.8, SD=0.05 -> a≒51.2, b≒12.8)
    psa_params['qaly_cancer_I'] = np.random.beta(51.2, 12.8)

    return psa_params

# ==========================================
# 2. コホート初期化関数
# ==========================================
def initialize_cohort(sex, strategy_p=False):
    df = pd.DataFrame({
        'id': np.arange(N_COHORT),
        'age': np.random.choice(AGE_GROUPS, size=N_COHORT, p=AGE_DIST) + np.random.randint(0, 5, size=N_COHORT),
        'sex': np.full(N_COHORT, sex, dtype='object'),
        'insurance': np.full(N_COHORT, 'Unknown', dtype='object'),
        'nhi_old_rule': np.random.rand(N_COHORT) < 0.30,
        'pylori_status': np.full(N_COHORT, 'Uninfected', dtype='object'),
        'atrophy': np.full(N_COHORT, 'None', dtype='object'),
        'cancer_status': np.full(N_COHORT, 'None', dtype='object'),
        'cancer_stage': np.full(N_COHORT, 'None', dtype='object'),
        'is_dead': np.zeros(N_COHORT, dtype=bool),
        'death_cause': np.full(N_COHORT, 'None', dtype='object'),
        'years_lived': np.zeros(N_COHORT, dtype=float),
        'total_cost': np.zeros(N_COHORT, dtype=float),
        'total_qaly': np.zeros(N_COHORT, dtype=float),
        'screen_count': np.zeros(N_COHORT, dtype=int),
        'true_positive': np.zeros(N_COHORT, dtype=int),
        'false_positive': np.zeros(N_COHORT, dtype=int),
        'last_screen_age': np.full(N_COHORT, -1, dtype=int)
    })

    for base_age, probs in INSURANCE_DIST.items():
        mask = (df['age'] >= base_age) & (df['age'] < base_age + 5)
        n_mask = mask.sum()
        if n_mask > 0:
            p_norm = probs / np.sum(probs)
            df.loc[mask, 'insurance'] = np.random.choice(['Employee', 'Dependent', 'NHI'], size=n_mask, p=p_norm)

    birth_years = START_YEAR - df['age']
    inf_probs = np.array([get_h_pylori_prob(y) for y in birth_years])
    df.loc[np.random.rand(N_COHORT) < inf_probs, 'pylori_status'] = 'Infected'

    inf_mask = df['pylori_status'] == 'Infected'
    sev_probs = np.array([get_severe_atrophy_prob(a) for a in df['age']])
    rand_vals = np.random.rand(N_COHORT)

    severe_mask = inf_mask & (rand_vals < sev_probs)
    mild_mask = inf_mask & (rand_vals >= sev_probs)

    df.loc[severe_mask, 'atrophy'] = 'Severe'
    df.loc[mild_mask, 'atrophy'] = 'Mild'

    if strategy_p:
        eradicate_mask = inf_mask & (np.random.rand(N_COHORT) < 0.90)
        df.loc[eradicate_mask, 'pylori_status'] = 'Eradicated'
        df.loc[eradicate_mask, 'total_cost'] += COST_PYLORI_TEST + COST_ERADICATION
        df.loc[(df['age'] >= 20) & (~eradicate_mask), 'total_cost'] += COST_PYLORI_TEST * 0.8

    return df

# ==========================================
# 3. シミュレーション実行関数 (PSA対応)
# ==========================================
def run_simulation(cohort, strategy_name, psa_params): # ★引数にpsa_paramsを追加
    df = cohort.copy()

    # ★PSAパラメータの展開
    rate_screen_S = psa_params['rate_screen_S']
    rate_erad_impl = psa_params['rate_erad_implement']
    cost_screening = psa_params['cost_screening']
    qaly_w_cancer_I = psa_params['qaly_cancer_I']

    cost_cancer_trt = BASE_COST_CANCER_TRT.copy()
    cost_cancer_trt['I'] = psa_params['cost_cancer_I']

    for year in range(SIM_YEARS):
        current_age = df['age']
        alive_mask = ~df['is_dead']

        if not alive_mask.any():
            break

        gen_death_probs = np.array([get_general_death_prob(a, df['sex'].iloc[0]) for a in current_age])
        gen_death_mask = alive_mask & (np.random.rand(N_COHORT) < gen_death_probs)
        df.loc[gen_death_mask, 'is_dead'] = True
        df.loc[gen_death_mask, 'death_cause'] = 'General'
        alive_mask = ~df['is_dead']

        diag_mask = alive_mask & (df['cancer_status'] == 'Diagnosed')
        for stage in STAGES:
            stage_mask = diag_mask & (df['cancer_stage'] == stage)
            death_mask = stage_mask & (np.random.rand(N_COHORT) < ANNUAL_CANCER_DEATH_PROB[stage])
            df.loc[death_mask, 'is_dead'] = True
            df.loc[death_mask, 'death_cause'] = 'GastricCancer'
        alive_mask = ~df['is_dead']

        risk_none = alive_mask & (df['cancer_status'] == 'None')
        for status in ['Infected', 'Eradicated', 'Uninfected']:
            status_mask = risk_none & (df['pylori_status'] == status)
            inc_prob = ANNUAL_INCIDENCE[df['sex'].iloc[0]][status]
            inc_mask = status_mask & (np.random.rand(N_COHORT) < inc_prob)
            df.loc[inc_mask, 'cancer_status'] = 'Undiagnosed'

        screen_probs = np.zeros(N_COHORT)
        if strategy_name == 'H':
            emp_mask = alive_mask & (df['insurance'] == 'Employee') & (current_age >= 40) & (current_age <= 64)
            dep_mask = alive_mask & (df['insurance'] == 'Dependent') & (current_age >= 50) & (current_age <= 64) & ((current_age - df['last_screen_age']) >= 2)
            nhi_new_mask = alive_mask & (df['insurance'] == 'NHI') & (~df['nhi_old_rule']) & (current_age >= 50) & (current_age <= 64) & ((current_age - df['last_screen_age']) >= 2)
            nhi_old_mask = alive_mask & (df['insurance'] == 'NHI') & (df['nhi_old_rule']) & (current_age >= 40) & (current_age <= 64)
            over65_mask = alive_mask & (current_age >= 65) & ((current_age - df['last_screen_age']) >= 2)

            screen_probs[emp_mask] = 0.75
            screen_probs[dep_mask] = 0.45
            screen_probs[nhi_new_mask] = 0.35
            screen_probs[nhi_old_mask] = 0.35
            screen_probs[over65_mask] = 0.35
        elif strategy_name == 'S':
            target_mask = alive_mask & (current_age >= 50) & ((current_age - df['last_screen_age']) >= 2)
            screen_probs[target_mask] = rate_screen_S # ★PSAパラメータを使用
        elif strategy_name == 'P':
            sev_mask = alive_mask & (df['pylori_status'].isin(['Infected', 'Eradicated'])) & (df['atrophy'] == 'Severe')
            mild_mask = alive_mask & (df['pylori_status'].isin(['Infected', 'Eradicated'])) & (df['atrophy'] == 'Mild') & ((current_age - df['last_screen_age']) >= 2)
            screen_probs[sev_mask] = 0.75
            screen_probs[mild_mask] = 0.75

        do_screen_mask = alive_mask & (np.random.rand(N_COHORT) < screen_probs)
        df.loc[do_screen_mask, 'screen_count'] += 1
        df.loc[do_screen_mask, 'last_screen_age'] = current_age
        df.loc[do_screen_mask, 'total_cost'] += cost_screening # ★PSAパラメータを使用

        if strategy_name == 'H':
            # ★PSAパラメータ(rate_erad_impl)を使用
            h_erad_mask = do_screen_mask & emp_mask & (df['pylori_status'] == 'Infected') & (np.random.rand(N_COHORT) < rate_erad_impl * 0.90)
            df.loc[h_erad_mask, 'pylori_status'] = 'Eradicated'
            df.loc[h_erad_mask, 'total_cost'] += COST_ERADICATION

        undiag_mask = alive_mask & (df['cancer_status'] == 'Undiagnosed')
        screen_found_mask = undiag_mask & do_screen_mask
        if screen_found_mask.any():
            df.loc[screen_found_mask, 'cancer_status'] = 'Diagnosed'
            df.loc[screen_found_mask, 'true_positive'] += 1
            n_found = screen_found_mask.sum()
            df.loc[screen_found_mask, 'cancer_stage'] = np.random.choice(STAGES, size=n_found, p=STAGE_DIST_SCREEN)

        fp_mask = alive_mask & (df['cancer_status'] == 'None') & do_screen_mask & (np.random.rand(N_COHORT) < 0.05)
        df.loc[fp_mask, 'false_positive'] += 1

        symp_found_mask = undiag_mask & (~do_screen_mask) & (np.random.rand(N_COHORT) < 0.20)
        if symp_found_mask.any():
            df.loc[symp_found_mask, 'cancer_status'] = 'Diagnosed'
            n_found = symp_found_mask.sum()
            df.loc[symp_found_mask, 'cancer_stage'] = np.random.choice(STAGES, size=n_found, p=STAGE_DIST_SYMPTOM)

        alive_mask = ~df['is_dead']
        df.loc[alive_mask, 'years_lived'] += 1
        df.loc[alive_mask, 'age'] += 1

        qaly_vals = np.ones(N_COHORT) * BASE_QALY_WEIGHTS['Healthy']
        for stage in STAGES:
            stage_mask = alive_mask & (df['cancer_status'] == 'Diagnosed') & (df['cancer_stage'] == stage)
            # ★PSAパラメータを使用
            q_weight = qaly_w_cancer_I if stage == 'I' else BASE_QALY_WEIGHTS[f'Cancer_{stage}']
            qaly_vals[stage_mask] = q_weight
            df.loc[stage_mask, 'total_cost'] += cost_cancer_trt[stage] / 5.0

        df.loc[alive_mask, 'total_qaly'] += qaly_vals[alive_mask]

    return df

# ==========================================
# 4. 反復シミュレーション用の指標計算と集計
# ==========================================
def calculate_iteration_metrics(results_dict, psa_params):
    metrics = {}
    # ★実行時のPSAパラメータを記録（散布図などの分析用）
    for k, v in psa_params.items():
        metrics[f'Param_{k}'] = v

    for st, df in results_dict.items():
        gc_deaths = (df['death_cause'] == 'GastricCancer').sum()
        person_years = df['years_lived'].sum()
        avg_cost = df['total_cost'].mean()
        avg_qaly = df['total_qaly'].mean()

        metrics[f'{st}_Deaths'] = gc_deaths
        metrics[f'{st}_PersonYears'] = person_years
        metrics[f'{st}_Cost'] = avg_cost
        metrics[f'{st}_QALY'] = avg_qaly
        metrics[f'{st}_Screens'] = df['screen_count'].sum()

    base_deaths = metrics['H_Deaths']
    base_cost = metrics['H_Cost']
    base_qaly = metrics['H_QALY']
    base_py = metrics['H_PersonYears']

    for st in ['S', 'P']:
        st_deaths = metrics[f'{st}_Deaths']
        st_cost = metrics[f'{st}_Cost']
        st_qaly = metrics[f'{st}_QALY']
        st_py = metrics[f'{st}_PersonYears']

        rr = (st_deaths / N_COHORT) / (base_deaths / N_COHORT) if base_deaths > 0 else np.nan
        metrics[f'{st}_vs_H_RR'] = rr

        rate_st = st_deaths / st_py if st_py > 0 else 0
        rate_h = base_deaths / base_py if base_py > 0 else 0
        hr_approx = rate_st / rate_h if rate_h > 0 else np.nan
        metrics[f'{st}_vs_H_HR_approx'] = hr_approx

        dc = st_cost - base_cost
        dq = st_qaly - base_qaly
        icer = dc / dq if dq > 0 else np.nan
        metrics[f'{st}_vs_H_ICER'] = icer
        metrics[f'{st}_vs_H_dCost'] = dc
        metrics[f'{st}_vs_H_dQALY'] = dq

    return metrics

def summarize_iterations(metrics_list, sex):
    df_res = pd.DataFrame(metrics_list)

    print(f"\n{'='*60}")
    print(f" PSA(確率的感度分析) 集計結果: {sex}")
    print(f" (コホート: {N_COHORT:,}人, サンプリング反復回数: {N_ITERATIONS}回)")
    print(f"{'='*60}")

    print("\n--- 胃がん死亡数 (平均 [95%信用区間]) ---")
    for st in ['H', 'S', 'P']:
        mean_val = df_res[f'{st}_Deaths'].mean()
        p025 = df_res[f'{st}_Deaths'].quantile(0.025)
        p975 = df_res[f'{st}_Deaths'].quantile(0.975)
        print(f"戦略{st}: {mean_val:.1f}人 [{p025:.1f} - {p975:.1f}]")

    print("\n--- RR: 相対リスク (平均 [95%信用区間]) ---")
    for st in ['S', 'P']:
        col = f'{st}_vs_H_RR'
        mean_val = df_res[col].mean()
        p025 = df_res[col].quantile(0.025)
        p975 = df_res[col].quantile(0.975)
        print(f"戦略{st} vs H: {mean_val:.3f} [{p025:.3f} - {p975:.3f}]")

    print("\n--- ICER: 増分費用効果比 (中央値 [95%信用区間]) JPY/QALY ---")
    for st in ['S', 'P']:
        icer_col = f'{st}_vs_H_ICER'
        dq_col = f'{st}_vs_H_dQALY'

        valid_icers = df_res[df_res[dq_col] > 0][icer_col].dropna()
        n_dominated = (df_res[dq_col] <= 0).sum()

        if len(valid_icers) > 0:
            median_val = valid_icers.median()
            p025 = valid_icers.quantile(0.025)
            p975 = valid_icers.quantile(0.975)
            print(f"戦略{st} vs H: {median_val:,.0f} [{p025:,.0f} - {p975:,.0f}]")
        else:
            print(f"戦略{st} vs H: 評価不能 (全回で Dominant または Dominated)")

        if n_dominated > 0:
            print(f"  ※ {n_dominated}回/{N_ITERATIONS}回 で戦略{st}はDominant(低コスト・高QALY)またはDominatedでした。")

# ==========================================
# メイン実行ブロック
# ==========================================
if __name__ == "__main__":
    np.random.seed(42) # 全体の再現性を担保

    for sex in ['Male', 'Female']:
        metrics_history = []
        print(f"\n>>> {sex} のPSAシミュレーションを開始します...")

        for i in range(N_ITERATIONS):
            # ★各イテレーションでパラメータを確率分布からサンプリング
            current_psa_params = sample_psa_parameters()

            base_cohort_h_s = initialize_cohort(sex, strategy_p=False)
            base_cohort_p = initialize_cohort(sex, strategy_p=True)

            results = {}
            # サンプリングしたパラメータをシミュレーション関数に渡す
            results['H'] = run_simulation(base_cohort_h_s, 'H', current_psa_params)
            results['S'] = run_simulation(base_cohort_h_s, 'S', current_psa_params)
            results['P'] = run_simulation(base_cohort_p, 'P', current_psa_params)

            # 結果と使用したパラメータを記録
            iteration_metrics = calculate_iteration_metrics(results, current_psa_params)
            iteration_metrics['Iteration'] = i + 1
            metrics_history.append(iteration_metrics)

        summarize_iterations(metrics_history, sex)


>>> Male のPSAシミュレーションを開始します...

 PSA(確率的感度分析) 集計結果: Male
 (コホート: 10,000人, サンプリング反復回数: 2回)

--- 胃がん死亡数 (平均 [95%信用区間]) ---
戦略H: 95.0人 [92.2 - 97.8]
戦略S: 123.0人 [116.3 - 129.7]
戦略P: 65.5人 [58.4 - 72.6]

--- RR: 相対リスク (平均 [95%信用区間]) ---
戦略S vs H: 1.294 [1.263 - 1.325]
戦略P vs H: 0.688 [0.633 - 0.742]

--- ICER: 増分費用効果比 (中央値 [95%信用区間]) JPY/QALY ---
戦略S vs H: 評価不能 (全回で Dominant または Dominated)
  ※ 2回/2回 で戦略SはDominant(低コスト・高QALY)またはDominatedでした。
戦略P vs H: 評価不能 (全回で Dominant または Dominated)
  ※ 2回/2回 で戦略PはDominant(低コスト・高QALY)またはDominatedでした。

>>> Female のPSAシミュレーションを開始します...

 PSA(確率的感度分析) 集計結果: Female
 (コホート: 10,000人, サンプリング反復回数: 2回)

--- 胃がん死亡数 (平均 [95%信用区間]) ---
戦略H: 62.5人 [61.1 - 63.9]
戦略S: 62.5人 [62.0 - 63.0]
戦略P: 31.5人 [25.3 - 37.7]

--- RR: 相対リスク (平均 [95%信用区間]) ---
戦略S vs H: 1.000 [0.985 - 1.016]
戦略P vs H: 0.507 [0.396 - 0.617]

--- ICER: 増分費用効果比 (中央値 [95%信用区間]) JPY/QALY ---
戦略S vs H: 726,006 [354,405 - 1,097,607]
戦略P vs H: -437,890 [-473,249 - -402,531]
